# Fold 4 - untouched 2024 validation

This notebook is a reader-facing audit of the single frozen Fold 4 execution. It reads the archived outputs only; it does not generate alerts, tune rules, or access 2025 results.

## tl;dr

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
OUT = ROOT / 'outputs' / 'role_validation' / 'fold_4'
run = json.loads((OUT / 'run_manifest.json').read_text())
assert run['test_season'] == 2024 and not run['2025_results_used']
assert run['seasons_admitted_to_feature_generation'] == [2024]
recommendations = pd.read_csv(OUT / 'fold4_family_recommendations.csv')
gates = pd.read_csv(OUT / 'fold4_gate_decisions.csv')
display(gates[['role_family','fold4_candidate_status','failed_checks']], recommendations)

,role_family,fold4_candidate_status,failed_checks
0,rb_carry_share,FAILS_FOLD_4_POINT_GATES,min_holdout_alerts
1,rb_opportunity_share,FAILS_FOLD_4_POINT_GATES,max_immediate_reversion_rate | direction_consi...
2,wr_target_share,NOT_APPLICABLE_RETIRED,retired_before_fold4
3,te_target_share,NOT_APPLICABLE_RETIRED,retired_before_fold4


,role_family,fold4_status,recommendation
0,rb_carry_share,FAILS_FOLD_4_POINT_GATES,CONTINUE_SHADOW_ONLY
1,rb_opportunity_share,FAILS_FOLD_4_POINT_GATES,CONTINUE_SHADOW_ONLY_WITHOUT_HOLDOUT_CLAIM
2,wr_target_share,NOT_APPLICABLE_RETIRED,REMAIN_RETIRED
3,te_target_share,NOT_APPLICABLE_RETIRED,REMAIN_RETIRED


## Context & Methods

### Key Assumptions

- The candidate and all release gates are byte-frozen.
- Confirmed partial games are excluded and suspected partial games are included in the primary policy.
- Comparators are equal-volume within every active-family week.
- Pooled metrics use concatenated raw alert rows, not seasonal averages.

In [2]:
frozen = json.loads((OUT / 'frozen_execution_package_manifest.json').read_text())
pre = json.loads((OUT / 'pre_run_manifest.json').read_text())
assert frozen['candidate_config_sha256'] == '4dcf389a1f8fcdd11a9277305a8372fadaabaa830185e07eff5d8fbb274a81c7'
assert pre['seasons_admitted_to_alert_selection'] == [2024]
assert pre['seasons_admitted_to_outcome_evaluation'] == [2024]
pd.DataFrame([{
    'execution_package_commit': frozen['execution_package_commit'],
    'physically_opened': pre['source_seasons_physically_opened'],
    'feature_seasons': pre['seasons_admitted_to_feature_generation'],
    'alert_seasons': pre['seasons_admitted_to_alert_selection'],
    'outcome_seasons': pre['seasons_admitted_to_outcome_evaluation'],
}])

,execution_package_commit,physically_opened,feature_seasons,alert_seasons,outcome_seasons
0,6446d4ef7fa554e20978c90fda6ddefbedafc4fa,"[2017, 2018, 2019, 2020, 2021, 2022, 2023, 202...",[2024],[2024],[2024]


## Data

In [3]:
audit = pd.read_csv(OUT / 'data_audit_2024.csv')
audit_checks = pd.read_csv(OUT / 'data_audit_checks_2024.csv')
joins = pd.read_csv(OUT / 'join_coverage_2024.csv')
assert audit.at[0, 'duplicate_key_rows'] == 0
assert audit.at[0, 'required_null_cells'] == 0
assert bool(audit_checks['passed'].all())
assert bool(joins['coverage_rate'].eq(1.0).all())
display(audit, audit_checks, joins)

,season,canonical_rows,unique_players,played_games,observed_weeks,duplicate_key_rows,duplicate_key_rate,required_null_cells,required_null_rows,identity_resolved_rows,identity_coverage,quality_pass_rows,quality_pass_rate,qualifying_rows,qualifying_rate
0,2024,7390,544,272,18,0,0.0,0,0,7390,1.0,7390,1.0,7390,1.0


,check,passed
0,canonical_grain_unique,True
1,required_fields_complete,True
2,identity_coverage_complete,True
3,played_weeks_complete,True
4,source_schema_and_games_complete,True
5,identity_and_opportunity_joins_complete,True
6,execution_hashes_verified,True
7,temporal_precheck_passed,True


,season,join,rows,matched_rows,coverage_rate
0,2024,opportunity_to_identity,31184,31184,1.0
1,2024,participating_player_to_identity,10209,10209,1.0


## Results

In [4]:
alerts = pd.read_csv(OUT / 'fold4_alerts_2024.csv.gz', low_memory=False)
primary = alerts.query("partial_policy == 'PRIMARY_CONFIRMED_EXCLUDED'")
rows = []
for (family, method), group in primary.groupby(['role_family','method']):
    evaluable = group['persistent'].notna()
    reversion_evaluable = group['immediate_reversion'].notna()
    rows.append({
        'role_family': family, 'method': method, 'alerts': len(group),
        'evaluable_alerts': int(evaluable.sum()),
        'persistent_alerts': int(group.loc[evaluable,'persistent'].astype(float).sum()),
        'precision': group.loc[evaluable,'persistent'].astype(float).mean(),
        'reversion_rate': group.loc[reversion_evaluable,'immediate_reversion'].astype(float).mean(),
        'median_retention': group.loc[evaluable,'retention'].median(),
    })
raw_recomputed = pd.DataFrame(rows)
stored = pd.read_csv(OUT / 'active_family_method_results_2024.csv').query("partial_policy == 'PRIMARY_CONFIRMED_EXCLUDED'")
merged = stored.merge(raw_recomputed,on=['role_family','method'],suffixes=('_stored','_raw'),validate='one_to_one')
for metric in ['alerts','evaluable_alerts','persistent_alerts','precision','reversion_rate','median_retention']:
    assert np.allclose(merged[f'{metric}_stored'],merged[f'{metric}_raw'],equal_nan=True)
display(raw_recomputed.sort_values(['role_family','method']))

,role_family,method,alerts,evaluable_alerts,persistent_alerts,precision,reversion_rate,median_retention
0,rb_carry_share,full_propwar,48,35,22,0.628571,0.200000,0.666747
1,rb_carry_share,naive_spike,48,35,14,0.400000,0.325000,0.372860
2,rb_carry_share,normal_game_trend,48,35,20,0.571429,0.190476,0.646291
3,rb_carry_share,two_week_raw,48,36,21,0.583333,0.219512,0.633804
4,rb_opportunity_share,full_propwar,55,48,29,0.604167,0.274510,0.785664
5,rb_opportunity_share,naive_spike,55,49,19,0.387755,0.392157,0.381605
6,rb_opportunity_share,normal_game_trend,55,47,27,0.574468,0.250000,0.662217
7,rb_opportunity_share,two_week_raw,55,48,30,0.625000,0.254902,0.663796


In [5]:
equal = pd.read_csv(OUT / 'equal_volume_verification_2024.csv')
temporal = pd.read_csv(OUT / 'temporal_integrity_checks_2024.csv')
direction = pd.read_csv(OUT / 'direction_results_2024.csv')
sensitivity = pd.read_csv(OUT / 'partial_game_sensitivity_2024.csv')
assert len(equal) == 108 and bool(equal['equal_volume'].all())
assert bool(temporal['passed'].all())
display(direction.query("partial_policy == 'PRIMARY_CONFIRMED_EXCLUDED'"), sensitivity)

,partial_policy,role_family,method,direction,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks
16,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,full_propwar,decrease,25,19,0.760000,14,0.736842,24,2,0.083333,0.749095,0.670925,22,12
17,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,full_propwar,increase,23,16,0.695652,8,0.500000,21,7,0.333333,0.417591,0.445157,19,11
18,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,naive_spike,decrease,23,19,0.826087,11,0.578947,20,7,0.350000,0.568794,0.560961,19,10
19,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,naive_spike,increase,25,16,0.640000,3,0.187500,20,6,0.300000,0.320560,0.288841,22,12
20,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,normal_game_trend,decrease,23,18,0.782609,15,0.833333,21,0,0.000000,0.814376,0.789932,17,11
21,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,normal_game_trend,increase,25,17,0.680000,5,0.294118,21,8,0.380952,0.260940,0.305868,20,11
22,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,two_week_raw,decrease,25,20,0.800000,15,0.750000,22,2,0.090909,0.794122,0.756767,18,12
23,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,two_week_raw,increase,23,16,0.695652,6,0.375000,19,7,0.368421,0.336153,0.397649,19,11
24,PRIMARY_CONFIRMED_EXCLUDED,rb_opportunity_share,full_propwar,decrease,33,27,0.818182,19,0.703704,29,6,0.206897,0.846994,0.594526,27,13
25,PRIMARY_CONFIRMED_EXCLUDED,rb_opportunity_share,full_propwar,increase,22,21,0.954545,10,0.476190,22,8,0.363636,0.324885,0.533284,17,10


,partial_policy,role_family,full_alerts,naive_alerts,full_evaluable_alerts,naive_evaluable_alerts,full_precision,naive_precision,precision_improvement,relative_precision_improvement,...,full_median_retention,naive_median_retention,sensitivity_type,delta_vs_primary_full_alerts,delta_vs_primary_full_evaluable_alerts,delta_vs_primary_full_precision,delta_vs_primary_precision_improvement,delta_vs_primary_full_reversion_rate,delta_vs_primary_reversion_improvement,delta_vs_primary_full_median_retention
0,ALL_INCLUDED,rb_carry_share,49,49,36,36,0.611111,0.416667,0.194444,0.466667,...,0.636439,0.393230,confirmed_partial_inclusion_sensitivity,1,1,-0.017460,-0.034127,-0.004348,-0.003579,-0.030308
1,ALL_INCLUDED,rb_opportunity_share,57,57,50,51,0.600000,0.392157,0.207843,0.530000,...,0.785664,0.381605,confirmed_partial_inclusion_sensitivity,2,2,-0.004167,-0.008568,-0.010359,-0.004440,0.000000
2,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,48,48,35,35,0.628571,0.400000,0.228571,0.571429,...,0.666747,0.372860,primary,0,0,0.000000,0.000000,0.000000,0.000000,0.000000
3,PRIMARY_CONFIRMED_EXCLUDED,rb_opportunity_share,55,55,48,49,0.604167,0.387755,0.216412,0.558114,...,0.785664,0.381605,primary,0,0,0.000000,0.000000,0.000000,0.000000,0.000000
4,STRICT_SUSPECTED_EXCLUDED,rb_carry_share,48,48,35,35,0.600000,0.342857,0.257143,0.750000,...,0.621503,0.335968,suspected_partial_exclusion_sensitivity,0,0,-0.028571,0.028571,0.050000,-0.057927,-0.045244
5,STRICT_SUSPECTED_EXCLUDED,rb_opportunity_share,55,55,49,50,0.612245,0.380000,0.232245,0.611171,...,0.717929,0.337873,suspected_partial_exclusion_sensitivity,0,1,0.008078,0.015833,-0.058824,0.019608,-0.067734


In [6]:
cross = pd.read_csv(OUT / 'cross_season_family_2021_2024.csv')
pooled_23 = pd.read_csv(OUT / 'pooled_untouched_family_2022_2023.csv')
pooled_24 = pd.read_csv(OUT / 'pooled_untouched_family_2022_2024.csv')
statuses = pd.read_csv(OUT / 'individual_season_gate_status_2021_2024.csv')
assert set(cross['period']) == {'redeveloped_2021','untouched_2022','untouched_2023','untouched_2024'}
display(cross, pooled_23, pooled_24, statuses)

,period,role_family,full_alerts,naive_alerts,full_evaluable_alerts,naive_evaluable_alerts,full_precision,naive_precision,precision_improvement,relative_precision_improvement,precision_improvement_ci_low,precision_improvement_ci_high,full_reversion_rate,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention,precision_ci_low,precision_ci_high
0,redeveloped_2021,rb_carry_share,56,56,43,41,0.744186,0.487805,0.256381,0.525581,0.006882,0.483918,0.187500,0.297872,0.110372,0.761506,0.404684,0.604651,0.860465
1,redeveloped_2021,rb_opportunity_share,77,77,55,55,0.672727,0.527273,0.145455,0.275862,-0.021458,0.277018,0.190476,0.301587,0.111111,0.684435,0.627807,0.545455,0.781818
2,untouched_2022,rb_carry_share,49,49,39,38,0.641026,0.473684,0.167341,0.353276,-0.008185,0.332159,0.150000,0.400000,0.250000,0.616781,0.388108,0.487179,0.794872
3,untouched_2022,rb_opportunity_share,59,59,47,45,0.617021,0.533333,0.083688,0.156915,-0.124477,0.280934,0.140000,0.319149,0.179149,0.687722,0.562135,0.468085,0.744681
4,untouched_2023,rb_carry_share,60,60,47,49,0.659574,0.530612,0.128962,0.243044,-0.030642,0.263314,0.230769,0.326923,0.096154,0.794931,0.526657,0.531915,0.787234
5,untouched_2023,rb_opportunity_share,74,74,57,58,0.771930,0.568966,0.202964,0.356725,0.057772,0.344047,0.156250,0.317460,0.161210,0.910085,0.616665,0.666667,0.877193
6,untouched_2024,rb_carry_share,48,48,35,35,0.628571,0.400000,0.228571,0.571429,0.054566,0.414287,0.200000,0.325000,0.125000,0.666747,0.372860,0.457143,0.771429
7,untouched_2024,rb_opportunity_share,55,55,48,49,0.604167,0.387755,0.216412,0.558114,-0.012212,0.438955,0.274510,0.392157,0.117647,0.785664,0.381605,0.458333,0.729167


,period,role_family,full_alerts,naive_alerts,full_evaluable_alerts,naive_evaluable_alerts,full_precision,naive_precision,precision_improvement,relative_precision_improvement,precision_improvement_ci_low,precision_improvement_ci_high,full_reversion_rate,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention,precision_ci_low,precision_ci_high
0,pooled_untouched_2022_2023,rb_carry_share,109,109,86,87,0.651163,0.505747,0.145416,0.287526,0.035280,0.249333,0.195652,0.358696,0.163043,0.738639,0.504611,0.546512,0.744186
1,pooled_untouched_2022_2023,rb_opportunity_share,133,133,104,103,0.701923,0.553398,0.148525,0.268387,0.023946,0.261304,0.149123,0.318182,0.169059,0.793768,0.587650,0.615385,0.788462


,period,role_family,full_alerts,naive_alerts,full_evaluable_alerts,naive_evaluable_alerts,full_precision,naive_precision,precision_improvement,relative_precision_improvement,precision_improvement_ci_low,precision_improvement_ci_high,full_reversion_rate,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention,precision_ci_low,precision_ci_high
0,pooled_untouched_2022_2024,rb_carry_share,157,157,121,122,0.644628,0.47541,0.169218,0.355942,0.082016,0.261744,0.197080,0.348485,0.151405,0.729678,0.435412,0.553719,0.727273
1,pooled_untouched_2022_2024,rb_opportunity_share,188,188,152,152,0.671053,0.50000,0.171053,0.342105,0.061092,0.283062,0.187879,0.341615,0.153736,0.793768,0.500988,0.598520,0.743421


,period,season,role_family,archived_status,frozen_before_holdout,interpretation
0,redeveloped_2021,2021,rb_carry_share,POINT_GATES_PASS,False,development diagnostic; not untouched
1,redeveloped_2021,2021,rb_opportunity_share,POINT_GATES_PASS,False,development diagnostic; not untouched
2,untouched_2022,2022,rb_carry_share,FAILS_FOLD_2_POINT_GATES,True,preserved archived Fold 2 decision
3,untouched_2022,2022,rb_opportunity_share,FAILS_FOLD_2_POINT_GATES,True,preserved archived Fold 2 decision
4,untouched_2023,2023,rb_carry_share,PASSES_FOLD_3_POINT_GATES,True,preserved archived Fold 3 decision
5,untouched_2023,2023,rb_opportunity_share,FAILS_FOLD_3_POINT_GATES,True,preserved archived Fold 3 decision
6,untouched_2024,2024,rb_carry_share,FAILS_FOLD_4_POINT_GATES,True,Fold 4 locked point-gate decision
7,untouched_2024,2024,rb_opportunity_share,FAILS_FOLD_4_POINT_GATES,True,Fold 4 locked point-gate decision


## Takeaways

In [7]:
comparisons = pd.read_csv(OUT / 'active_family_comparisons_2024.csv').query("partial_policy == 'PRIMARY_CONFIRMED_EXCLUDED'")
weekly = pd.read_csv(OUT / 'weekly_stability_2024.csv').query("partial_policy == 'PRIMARY_CONFIRMED_EXCLUDED' and method == 'full_propwar'")
summary = comparisons[['role_family','full_alerts','full_evaluable_alerts','full_precision','naive_precision','precision_improvement','full_reversion_rate','reversion_improvement','full_median_retention']].merge(
    gates[['role_family','fold4_candidate_status']], on='role_family', validate='one_to_one'
).merge(recommendations[['role_family','recommendation']],on='role_family',how='left')
display(summary, weekly[['role_family','weekly_median','weekly_maximum','zero_alert_weeks']])
print('These point-gate decisions do not constitute validation. No 2025 result was used.')

,role_family,full_alerts,full_evaluable_alerts,full_precision,naive_precision,precision_improvement,full_reversion_rate,reversion_improvement,full_median_retention,fold4_candidate_status,recommendation
0,rb_carry_share,48,35,0.628571,0.400000,0.228571,0.20000,0.125000,0.666747,FAILS_FOLD_4_POINT_GATES,CONTINUE_SHADOW_ONLY
1,rb_opportunity_share,55,48,0.604167,0.387755,0.216412,0.27451,0.117647,0.785664,FAILS_FOLD_4_POINT_GATES,CONTINUE_SHADOW_ONLY_WITHOUT_HOLDOUT_CLAIM


,role_family,weekly_median,weekly_maximum,zero_alert_weeks
8,rb_carry_share,3.0,7,5
12,rb_opportunity_share,3.0,8,5


These point-gate decisions do not constitute validation. No 2025 result was used.
